In [31]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix


In [17]:
columns = [
    'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
    'savings_status', 'employment', 'installment_commitment', 'personal_status',
    'other_parties', 'residence_since', 'property_magnitude', 'age',
    'other_payment_plans', 'housing', 'existing_credits', 'job',
    'num_dependents', 'own_telephone', 'foreign_worker', 'class'
]
df = pd.read_csv('german.data',sep=' ', header=None, names=columns)
df.head(5)

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   checking_status         1000 non-null   str  
 1   duration                1000 non-null   int64
 2   credit_history          1000 non-null   str  
 3   purpose                 1000 non-null   str  
 4   credit_amount           1000 non-null   int64
 5   savings_status          1000 non-null   str  
 6   employment              1000 non-null   str  
 7   installment_commitment  1000 non-null   int64
 8   personal_status         1000 non-null   str  
 9   other_parties           1000 non-null   str  
 10  residence_since         1000 non-null   int64
 11  property_magnitude      1000 non-null   str  
 12  age                     1000 non-null   int64
 13  other_payment_plans     1000 non-null   str  
 14  housing                 1000 non-null   str  
 15  existing_credits        1000 non-

In [19]:
col = ['checking_status', 'savings_status', 'property_magnitude']
for i in col:
    print(f'value counts for {i}:')
    print(df[i].value_counts())
    print()

value counts for checking_status:
checking_status
A14    394
A11    274
A12    269
A13     63
Name: count, dtype: int64

value counts for savings_status:
savings_status
A61    603
A65    183
A62    103
A63     63
A64     48
Name: count, dtype: int64

value counts for property_magnitude:
property_magnitude
A123    332
A121    282
A122    232
A124    154
Name: count, dtype: int64



In [20]:
mappings = {
    'checking_status': {
        'A11': '< 0 DM', 'A12': '0-200 DM', 'A13': '>= 200 DM', 'A14': 'no checking account'
    },
    'credit_history': {
        'A30': 'no credits/all paid duly', 'A31': 'all credits at this bank paid duly',
        'A32': 'existing credits paid duly', 'A33': 'delay in past', 'A34': 'critical/other credits existing'
    },
    'purpose': {
        'A40': 'car (new)', 'A41': 'car (used)', 'A42': 'furniture/equipment', 'A43': 'radio/tv',
        'A44': 'domestic appliances', 'A45': 'repairs', 'A46': 'education', 'A48': 'retraining',
        'A49': 'business', 'A410': 'others'
    },
    'savings_status': {
        'A61': '< 100 DM', 'A62': '100-500 DM', 'A63': '500-1000 DM',
        'A64': '>= 1000 DM', 'A65': 'unknown/no savings account'
    },
    'employment': {
        'A71': 'unemployed', 'A72': '< 1 year', 'A73': '1-4 years',
        'A74': '4-7 years', 'A75': '>= 7 years'
    },
    'personal_status': {
        'A91': 'male divorced/separated', 'A92': 'female divorced/separated/married',
        'A93': 'male single', 'A94': 'male married/widowed', 'A95': 'female single'
    },
    'other_parties': {
        'A101': 'none', 'A102': 'co-applicant', 'A103': 'guarantor'
    },
    'property_magnitude': {
        'A121': 'real estate', 'A122': 'building society savings/life insurance',
        'A123': 'car or other', 'A124': 'unknown/no property'
    },
    'other_payment_plans': {
        'A141': 'bank', 'A142': 'stores', 'A143': 'none'
    },
    'housing': {
        'A151': 'rent', 'A152': 'own', 'A153': 'for free'
    },
    'job': {
        'A171': 'unemployed/unskilled non-resident', 'A172': 'unskilled resident',
        'A173': 'skilled employee', 'A174': 'management/self-employed/highly qualified'
    },
    'own_telephone': {
        'A191': 'none', 'A192': 'yes'
    },
    'foreign_worker': {
        'A201': 'yes', 'A202': 'no'
    },
    'class': {
        1: 'good', 2: 'bad'
    }
}

df_readable = df.copy()
for col, mapping in mappings.items():
    df_readable[col] = df_readable[col].map(mapping)

In [21]:
mnar_cols = ['checking_status', 'savings_status', 'property_magnitude']

for col in mnar_cols:
    print(f'--- {col} vs class ---')
    print(df_readable.groupby(col)['class'].value_counts(normalize=True))
    print()

--- checking_status vs class ---
checking_status      class
0-200 DM             good     0.609665
                     bad      0.390335
< 0 DM               good     0.507299
                     bad      0.492701
>= 200 DM            good     0.777778
                     bad      0.222222
no checking account  good     0.883249
                     bad      0.116751
Name: proportion, dtype: float64

--- savings_status vs class ---
savings_status              class
100-500 DM                  good     0.669903
                            bad      0.330097
500-1000 DM                 good     0.825397
                            bad      0.174603
< 100 DM                    good     0.640133
                            bad      0.359867
>= 1000 DM                  good     0.875000
                            bad      0.125000
unknown/no savings account  good     0.825137
                            bad      0.174863
Name: proportion, dtype: float64

--- property_magnitude vs class --

In [22]:
checking_order = ['no checking account', '>= 200 DM', '0-200 DM', '< 0 DM']  # rendah -> tinggi risiko
saving_order = ['>= 1000 DM', 'unknown/no savings account', '500-1000 DM', '100-500 DM', '< 100 DM']
property_order = ['real estate', 'building society savings/life insurance', 'car or other', 'unknown/no property']


In [23]:
df['personal_status'].value_counts()

personal_status
A93    548
A92    310
A94     92
A91     50
Name: count, dtype: int64

In [24]:
df_readable['job'].value_counts()

job
skilled employee                             630
unskilled resident                           200
management/self-employed/highly qualified    148
unemployed/unskilled non-resident             22
Name: count, dtype: int64

In [25]:
employment_order =['unemployed', '< 1 year','1-4 years', '4-7 years', '>= 7 years']
credit_history_order = ['no credits/all paid duly', 'all credits at this bank paid duly','existing credits paid duly', 'delay in past', 'critical/other credits existing']
job_order = ['unemployed/unskilled non-resident', 'unskilled resident', 'skilled employee','management/self-employed/highly qualified']

In [26]:
ordinal_cols = ['checking_status', 'savings_status', 'property_magnitude', 'employment', 'credit_history', 'job']
nominal_cols =['purpose','housing','personal_status', 'other_parties', 'other_payment_plans', 'own_telephone', 'foreign_worker']
numerik_cols = ['duration', 'credit_amount', 'installment_commitment','residence_since', 'age', 'existing_credits', 'num_dependents']


In [27]:
ordibaordinal_categories = [checking_order, saving_order, property_order, employment_order, credit_history_order, job_order] #urutsannya harus dari index 0 dst
preprocessor = ColumnTransformer(transformers=[
    ('ord', OrdinalEncoder(categories=ordibaordinal_categories), ordinal_cols),
    ('nom', OneHotEncoder(handle_unknown='ignore'), nominal_cols),
    ('num', StandardScaler(), numerik_cols)
])

In [28]:
X = df_readable.drop(columns=['class'])
y = df_readable['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [30]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](2,)","['bad','good']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['checking_status','duration','credit_history',...,'num_dependents', 'own_telephone','foreign_worker']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ord', ...), ('nom', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'

In [32]:
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

         bad       0.52      0.47      0.49        60
        good       0.78      0.81      0.80       140

    accuracy                           0.71       200
   macro avg       0.65      0.64      0.64       200
weighted avg       0.70      0.71      0.71       200

[[ 28  32]
 [ 26 114]]


In [34]:
y_proba = pipeline.predict_proba(X_test)[:, 0]
y_proba

array([0.11139707, 0.80425581, 0.66904737, 0.06259868, 0.5433225 ,
       0.0982114 , 0.11576566, 0.46639545, 0.71950726, 0.24350482,
       0.13404891, 0.2820225 , 0.39057104, 0.68747665, 0.20130196,
       0.14844423, 0.23284129, 0.02094535, 0.11478345, 0.03276612,
       0.18217756, 0.08799086, 0.22524878, 0.05808933, 0.08636159,
       0.43243995, 0.14096152, 0.43234487, 0.12365457, 0.82944106,
       0.32650658, 0.14342962, 0.11806429, 0.14983488, 0.66558064,
       0.11303655, 0.16427105, 0.2024698 , 0.01110175, 0.80451036,
       0.01679121, 0.05525805, 0.00588365, 0.68968891, 0.76775826,
       0.93677301, 0.33650798, 0.12734826, 0.7495913 , 0.13644228,
       0.09900773, 0.29686458, 0.67962139, 0.02398461, 0.24614291,
       0.06671087, 0.02688364, 0.26803012, 0.01753486, 0.18787425,
       0.35689163, 0.36905959, 0.10643758, 0.0285413 , 0.87639538,
       0.11679962, 0.04809952, 0.04616507, 0.72022287, 0.05537179,
       0.08201558, 0.38833448, 0.10967884, 0.50333096, 0.04501

In [36]:
y_proba_bad = pipeline.predict_proba(X_test)[:, 0]

for threshold in [0.5, 0.4, 0.35, 0.3]:
    y_pred_thres = ['bad' if p >= threshold else 'good' for p in y_proba_bad]
    print(f'----Threshold = {threshold}----')
    print(classification_report(y_test, y_pred_thres))

----Threshold = 0.5----
              precision    recall  f1-score   support

         bad       0.52      0.47      0.49        60
        good       0.78      0.81      0.80       140

    accuracy                           0.71       200
   macro avg       0.65      0.64      0.64       200
weighted avg       0.70      0.71      0.71       200

----Threshold = 0.4----
              precision    recall  f1-score   support

         bad       0.52      0.57      0.54        60
        good       0.81      0.77      0.79       140

    accuracy                           0.71       200
   macro avg       0.66      0.67      0.66       200
weighted avg       0.72      0.71      0.71       200

----Threshold = 0.35----
              precision    recall  f1-score   support

         bad       0.52      0.68      0.59        60
        good       0.84      0.73      0.78       140

    accuracy                           0.71       200
   macro avg       0.68      0.71      0.69       200
w

In [37]:
for threshold in [0.5, 0.4, 0.35, 0.3]:
    y_pred_thresh = ['bad' if p >= threshold else 'good' for p in y_proba_bad]
    
    cm = confusion_matrix(y_test, y_pred_thresh, labels=['bad', 'good'])
    # cm[0][0]=actual bad, pred bad (TP)   | cm[0][1]=actual bad, pred good (FN)
    # cm[1][0]=actual good, pred bad (FP)  | cm[1][1]=actual good, pred good (TN)
    
    fn = cm[0][1]  # bad diprediksi good -> mahal (cost x5)
    fp = cm[1][0]  # good diprediksi bad -> murah (cost x1)
    
    total_cost = (fn * 5) + fp
    print(f'threshold={threshold} | FN={fn} FP={fp} | total_cost={total_cost}')

threshold=0.5 | FN=32 FP=26 | total_cost=186
threshold=0.4 | FN=26 FP=32 | total_cost=162
threshold=0.35 | FN=19 FP=38 | total_cost=133
threshold=0.3 | FN=17 FP=44 | total_cost=129


In [38]:
import numpy as np

thresholds = np.arange(0.05, 0.55, 0.01)
costs = []

for t in thresholds:
    y_pred_t = ['bad' if p >= t else 'good' for p in y_proba_bad]
    cm = confusion_matrix(y_test, y_pred_t, labels=['bad', 'good'])
    fn = cm[0][1]
    fp = cm[1][0]
    cost = (fn * 5) + fp
    costs.append(cost)

best_idx = np.argmin(costs)
print(f'Threshold optimal: {thresholds[best_idx]:.2f}, cost: {costs[best_idx]}')

Threshold optimal: 0.15, cost: 98


In [39]:
y_pred_015 = ['bad' if p >= 0.15 else 'good' for p in y_proba_bad]
print(classification_report(y_test, y_pred_015))

              precision    recall  f1-score   support

         bad       0.44      0.90      0.59        60
        good       0.92      0.51      0.66       140

    accuracy                           0.63       200
   macro avg       0.68      0.71      0.63       200
weighted avg       0.78      0.63      0.64       200



In [40]:
import joblib

joblib.dump(pipeline, 'german_credit_pipeline.pkl')

['german_credit_pipeline.pkl']

In [41]:
class CreditRiskPredictor:
    def __init__(self, pipeline_path, threshold=0.15):
        self.pipeline = joblib.load(pipeline_path)
        self.threshold = threshold
        self.mappings = mappings  # dict mapping A11->'< 0 DM' dst, yang sudah kamu buat sebelumnya

    def _to_readable(self, df_raw):
        """Convert kode asli (A11, A12, dst) ke human-readable, sama seperti training."""
        df_readable = df_raw.copy()
        for col, mapping in self.mappings.items():
            if col in df_readable.columns:
                df_readable[col] = df_readable[col].map(mapping)
        return df_readable

    def predict(self, df_raw):
        """
        df_raw: DataFrame dengan kolom kode asli (A11, A12, dst),
        format sama persis seperti german.data (belum di-mapping).
        """
        df_readable = self._to_readable(df_raw)

        # cek ada value yang gagal di-mapping (jadi NaN karena kode nggak dikenal)
        if df_readable.isnull().any().any():
            raise ValueError("Ada kode kategori yang tidak dikenali setelah mapping — cek input.")

        proba_bad = self.pipeline.predict_proba(df_readable)[:, 0]
        pred = ['bad' if p >= self.threshold else 'good' for p in proba_bad]

        return pd.DataFrame({
            'prob_bad': proba_bad,
            'prediction': pred
        })

In [43]:
predictor = CreditRiskPredictor('german_credit_pipeline.pkl', threshold=0.15)

# contoh 1 baris data baru, format kode asli (bukan readable)
raw_sample = df.loc[X_test.index[[0]]].drop(columns=['class'])

result = predictor.predict(raw_sample)
print(result)
print('Actual:', y_test.iloc[0]) 

   prob_bad prediction
0  0.111397       good
Actual: good


In [44]:
# ambil semua baris test set, tapi versi kode asli dari df
X_test_raw = df.loc[X_test.index].drop(columns=['class'])

# prediksi batch lewat predictor class
result = predictor.predict(X_test_raw)

# evaluasi — y_test sudah dalam bentuk 'good'/'bad' (dari df_readable)
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, result['prediction']))
print(confusion_matrix(y_test, result['prediction'], labels=['bad', 'good']))

              precision    recall  f1-score   support

         bad       0.44      0.90      0.59        60
        good       0.92      0.51      0.66       140

    accuracy                           0.63       200
   macro avg       0.68      0.71      0.63       200
weighted avg       0.78      0.63      0.64       200

[[54  6]
 [68 72]]
